In [ ]:
! conda install pandas ete3 legacy-cgi -c conda-forge -y

In [18]:
import pandas as pd
from ete3 import NCBITaxa
import sys
import datetime

# --- Configuration ---

def get_host_species(host_name, ncbi):
    """
    Checks a host name against the NCBI taxonomy database.

    Args:
        host_name (str): The name of the host to check (e.g., "Escherichia coli").
        ncbi (NCBITaxa): An instance of the ete3 NCBITaxa object.

    Returns:
        str: The species-level name if it meets all criteria (is Bacteria/Archaea,
             has a species level).
        None: If the host does not meet the criteria or is not found.
    """
    if not isinstance(host_name, str) or not host_name.strip():
        # Handle empty or non-string host names
        return None

    try:
        # 1. Get the TaxID for the given host name.
        # get_name_translator returns a dictionary like {'Escherichia coli': [940803]}
        name2taxid = ncbi.get_name_translator([host_name])
        if not name2taxid:
            print(f"Warning: Host '{host_name}' not found in NCBI taxonomy. Skipping.", file=sys.stderr)
            return None
        
        # We take the first TaxID if multiple are returned for a name
        taxid = name2taxid[host_name][0]

        # 2. Get the full taxonomic lineage (path from root to the TaxID).
        lineage_taxids = ncbi.get_lineage(taxid)
        if not lineage_taxids:
            return None # Should not happen if taxid is valid, but good practice

        # 3. Get the scientific names for all TaxIDs in the lineage.
        # This is more efficient than calling get_taxid_translator for each one.
        lineage_names_dict = ncbi.get_taxid_translator(lineage_taxids)
        lineage_names = list(lineage_names_dict.values())

        # 4. Check if the host belongs to "Bacteria" or "Archaea".
        is_prokaryote = "Bacteria" in lineage_names or "Archaea" in lineage_names
        if not is_prokaryote:
            # print(f"Info: Host '{host_name}' is not Bacteria or Archaea. Skipping.")
            return None

        # 5. Check for a species-level name and extract it.
        # Get the rank for each TaxID in the lineage.
        ranks = ncbi.get_rank(lineage_taxids)
        species_name = None
        for tax_id in lineage_taxids:
            if ranks.get(tax_id) == 'species':
                # We found the species level in the lineage.
                # Its name is in the dictionary we already fetched.
                species_name = lineage_names_dict.get(tax_id)
                break # Stop once the species level is found

        # 6. Return the species name if all conditions are met.
        if species_name:
            return species_name
        else:
            # print(f"Info: Host '{host_name}' is a prokaryote but has no species rank. Skipping.")
            return None

    except Exception as e:
        print(f"Error processing host '{host_name}': {e}. Skipping.", file=sys.stderr)
        return None


def main(INPUT_CSV_FILE, OUTPUT_CSV_FILE):
    """
    Main function to read, process, and write the CSV data.
    """
    print("Initializing NCBI Taxonomy database... (This may take a moment on first run)")
    # Initialize the NCBI Taxonomy object.
    # This will download the database if it doesn't exist locally.
    ncbi = NCBITaxa()
    # To force an update of the database, you can uncomment the next line:
    # ncbi.update_taxonomy_database()
    print("Database initialized.")

    try:
        # Read the input CSV file into a pandas DataFrame.
        df = pd.read_csv(INPUT_CSV_FILE)
        print(f"Successfully read '{INPUT_CSV_FILE}' with {len(df)} rows.")
    except FileNotFoundError:
        print(f"Error: The file '{INPUT_CSV_FILE}' was not found. Please check the filename and path.", file=sys.stderr)
        return
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}", file=sys.stderr)
        return

    # Ensure the required columns exist
    if 'Accession' not in df.columns or 'Host' not in df.columns:
        print(f"Error: The CSV file must contain 'Accession' and 'Host' columns.", file=sys.stderr)
        return

    results = []
    print("Processing hosts... This may take some time depending on the file size.")
    
    # Iterate over each row in the DataFrame
    for index, row in df.iterrows():
        accession = row['Accession']
        host_name = row['Host']
        
        # Process the host name using our function
        valid_species_name = get_host_species(host_name, ncbi)
        
        # If a valid species name was returned, add it to our results
        if valid_species_name:
            results.append({
                'Accession': accession,
                'Host': valid_species_name
            })
        
        # Optional: Print progress
        if (index + 1) % 100 == 0:
            print(f"Processed {index + 1}/{len(df)} rows...")

    # Create a new DataFrame from the collected results
    output_df = pd.DataFrame(results)
    today = datetime.date.today()
    if not output_df.empty:
        # Write the new DataFrame to the output CSV file
        # index=False prevents pandas from writing the DataFrame index as a column
        output_df.to_csv(f'{OUTPUT_CSV_FILE}_{today}.csv', index=False)
        print(f"\nProcessing complete. Found {len(output_df)} valid hosts.")
        print(f"Output saved to '{OUTPUT_CSV_FILE}'.")
    else:
        print("\nProcessing complete. No hosts met the specified criteria.")


In [19]:
main("sequences.csv", "RefSeq-VHDB")

Initializing NCBI Taxonomy database... (This may take a moment on first run)
Database initialized.
Successfully read 'sequences.csv' with 17727 rows.
Processing hosts... This may take some time depending on the file size.
Processed 100/17727 rows...
Processed 200/17727 rows...
Processed 300/17727 rows...
Processed 400/17727 rows...


Processed 500/17727 rows...
Processed 600/17727 rows...
Processed 700/17727 rows...
Processed 800/17727 rows...
Processed 900/17727 rows...
Processed 1000/17727 rows...
Processed 1100/17727 rows...
Processed 1200/17727 rows...
Processed 1300/17727 rows...
Processed 1400/17727 rows...
Processed 1500/17727 rows...


Processed 1600/17727 rows...
Processed 1700/17727 rows...
Processed 1800/17727 rows...
Processed 1900/17727 rows...
Processed 2000/17727 rows...
Processed 2100/17727 rows...
Processed 2200/17727 rows...
Processed 2300/17727 rows...
Processed 2400/17727 rows...
Processed 2500/17727 rows...
Processed 2600/17727 rows...
Processed 2700/17727 rows...
Processed 2800/17727 rows...
Processed 2900/17727 rows...
Processed 3000/17727 rows...
Processed 3100/17727 rows...
Processed 3200/17727 rows...
Processed 3300/17727 rows...
Processed 3400/17727 rows...
Processed 3500/17727 rows...


Processed 3600/17727 rows...
Processed 3700/17727 rows...
Processed 3800/17727 rows...
Processed 3900/17727 rows...
Processed 4000/17727 rows...
Processed 4100/17727 rows...
Processed 4200/17727 rows...
Processed 4300/17727 rows...
Processed 4400/17727 rows...
Processed 4500/17727 rows...
Processed 4600/17727 rows...
Processed 4700/17727 rows...
Processed 4800/17727 rows...
Processed 4900/17727 rows...
Processed 5000/17727 rows...
Processed 5100/17727 rows...
Processed 5200/17727 rows...


Processed 5300/17727 rows...
Processed 5400/17727 rows...
Processed 5500/17727 rows...
Processed 5600/17727 rows...
Processed 5700/17727 rows...
Processed 5800/17727 rows...
Processed 5900/17727 rows...
Processed 6000/17727 rows...
Processed 6100/17727 rows...
Processed 6200/17727 rows...
Processed 6300/17727 rows...
Processed 6400/17727 rows...
Processed 6500/17727 rows...
Processed 6600/17727 rows...
Processed 6700/17727 rows...
Processed 6800/17727 rows...
Processed 6900/17727 rows...
Processed 7000/17727 rows...
Processed 7100/17727 rows...
Processed 7200/17727 rows...


Processed 7300/17727 rows...
Processed 7400/17727 rows...
Processed 7500/17727 rows...
Processed 7600/17727 rows...
Processed 7700/17727 rows...
Processed 7800/17727 rows...
Processed 7900/17727 rows...
Processed 8000/17727 rows...
Processed 8100/17727 rows...
Processed 8200/17727 rows...
Processed 8300/17727 rows...
Processed 8400/17727 rows...
Processed 8500/17727 rows...
Processed 8600/17727 rows...
Processed 8700/17727 rows...
Processed 8800/17727 rows...
Processed 8900/17727 rows...


Processed 9000/17727 rows...
Processed 9100/17727 rows...
Processed 9200/17727 rows...
Processed 9300/17727 rows...
Processed 9400/17727 rows...
Processed 9500/17727 rows...
Processed 9600/17727 rows...
Processed 9700/17727 rows...
Processed 9800/17727 rows...
Processed 9900/17727 rows...
Processed 10000/17727 rows...
Processed 10100/17727 rows...
Processed 10200/17727 rows...
Processed 10300/17727 rows...
Processed 10400/17727 rows...
Processed 10500/17727 rows...
Processed 10600/17727 rows...
Processed 10700/17727 rows...
Processed 10800/17727 rows...
Processed 10900/17727 rows...
Processed 11000/17727 rows...
Processed 11100/17727 rows...


Processed 11200/17727 rows...
Processed 11300/17727 rows...
Processed 11400/17727 rows...
Processed 11500/17727 rows...
Processed 11600/17727 rows...
Processed 11700/17727 rows...
Processed 11800/17727 rows...
Processed 11900/17727 rows...
Processed 12000/17727 rows...
Processed 12100/17727 rows...
Processed 12200/17727 rows...
Processed 12300/17727 rows...
Processed 12400/17727 rows...
Processed 12500/17727 rows...
Processed 12600/17727 rows...
Processed 12700/17727 rows...
Processed 12800/17727 rows...
Processed 12900/17727 rows...
Processed 13000/17727 rows...
Processed 13100/17727 rows...
Processed 13200/17727 rows...
Processed 13300/17727 rows...
Processed 13400/17727 rows...
Processed 13500/17727 rows...
Processed 13600/17727 rows...
Processed 13700/17727 rows...
Processed 13800/17727 rows...
Processed 13900/17727 rows...
Processed 14000/17727 rows...
Processed 14100/17727 rows...
Processed 14200/17727 rows...


Processed 14300/17727 rows...
Processed 14400/17727 rows...
Processed 14500/17727 rows...
Processed 14600/17727 rows...
Processed 14700/17727 rows...
Processed 14800/17727 rows...
Processed 14900/17727 rows...
Processed 15000/17727 rows...
Processed 15100/17727 rows...
Processed 15200/17727 rows...
Processed 15300/17727 rows...
Processed 15400/17727 rows...
Processed 15500/17727 rows...
Processed 15600/17727 rows...
Processed 15700/17727 rows...
Processed 15800/17727 rows...
Processed 15900/17727 rows...
Processed 16000/17727 rows...
Processed 16100/17727 rows...
Processed 16200/17727 rows...
Processed 16300/17727 rows...
Processed 16400/17727 rows...
Processed 16500/17727 rows...
Processed 16600/17727 rows...


Processed 16700/17727 rows...
Processed 16800/17727 rows...
Processed 16900/17727 rows...
Processed 17000/17727 rows...
Processed 17100/17727 rows...
Processed 17200/17727 rows...
Processed 17300/17727 rows...
Processed 17400/17727 rows...
Processed 17500/17727 rows...
Processed 17600/17727 rows...
Processed 17700/17727 rows...

Processing complete. Found 4622 valid hosts.
Output saved to 'RefSeq-VHDB'.
